# Wishart Graph Dictionary — resumable CPU/CUDA Colab experiment

This notebook runs `feature/wishart-colab-gpu-resume` against the 100k graph-dictionary config stored on Google Drive. All heavy reads/writes and computation happen under `/content`; a complete state checkpoint is synchronized to Drive after each graph-contraction level.

**Important:** CUDA accelerates type-vector cosine kNN, not the CPU-bound VF2 isomorphism or sparse ego extraction. Colab GPU availability and speed vary. Re-running the notebook with the same `RUN_NAME` automatically resumes an interrupted, checkpointed run; an already completed run is never overwritten.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil, subprocess, sys, tempfile

BRANCH = 'feature/wishart-colab-gpu-resume'
REPO_URL = 'https://github.com/SemanticMap/semgraphex.git'
REPO_DIR = Path('/content/semgraphex')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

# semgraphex is private. Prefer a Colab Secret named GITHUB_TOKEN with repo read access.
# The token is never printed or written into the notebook.
public_probe = subprocess.run(
    ['git', 'ls-remote', REPO_URL, BRANCH],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
).returncode == 0

env = os.environ.copy()
askpass_path = None
if not public_probe:
    from google.colab import userdata
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception as error:
        raise RuntimeError(
            'Private GitHub access required. In Colab open the Secrets panel, '
            'add GITHUB_TOKEN with read access to SemanticMap/semgraphex, '
            'enable notebook access to that secret, then rerun this cell.'
        ) from error
    if not token:
        raise RuntimeError('Colab Secret GITHUB_TOKEN is empty.')
    fd, raw_path = tempfile.mkstemp(prefix='semmap-git-askpass-', suffix='.sh')
    os.close(fd)
    askpass_path = Path(raw_path)
    askpass_path.write_text(
        '#!/bin/sh\ncase "$1" in\n*Username*) echo "x-access-token" ;;\n*Password*) echo "$GITHUB_TOKEN" ;;\nesac\n',
        encoding='utf-8',
    )
    askpass_path.chmod(0o700)
    env['GIT_ASKPASS'] = str(askpass_path)
    env['GIT_TERMINAL_PROMPT'] = '0'
    env['GITHUB_TOKEN'] = token

try:
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
        env=env,
    )
finally:
    if askpass_path is not None:
        askpass_path.unlink(missing_ok=True)
    env.pop('GITHUB_TOKEN', None)

os.chdir(REPO_DIR)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print({'branch': BRANCH, 'commit': head, 'repo': str(REPO_DIR)})


In [ ]:
# Install the exact checked-out branch and Wishart/notebook dependencies.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-c', 'requirements/constraints.txt',
    '-e', '.[wishart,notebook]'
], check=True)
print('semmap-haken installed from', head)


In [ ]:
# Google OAuth confirmation is the only interactive step.
from google.colab import drive
drive.mount('/content/drive')


## Configuration, input and restart

`DRIVE_ROOT` is the only durable root. An editable `config/experiment.yaml` is copied there once from Git; subsequent runs read **that Drive copy**, not the Git template. To resume after a Colab reset, keep the same `RUN_NAME`, unchanged configuration and input data, and the same Git revision. The notebook selects the latest completed prepared graph in `prepared/` or a source TSV in `data/`. Set an absolute source override if the data are stored in an older Drive location.


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/SemanticMap/colab/wishart')
CONFIG_ON_DRIVE = DRIVE_ROOT / 'config' / 'experiment.yaml'
CONFIG_TEMPLATE = REPO_DIR / 'configs/wishart_conceptnet_dictionary.yaml'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG_ON_DRIVE.parent.mkdir(parents=True, exist_ok=True)
if not CONFIG_ON_DRIVE.is_file():
    shutil.copy2(CONFIG_TEMPLATE, CONFIG_ON_DRIVE)
    print('Created editable Drive config:', CONFIG_ON_DRIVE)
else:
    print('Reusing Drive config:', CONFIG_ON_DRIVE)

# Use a STABLE run name across Colab sessions. Change it to start a NEW experiment.
RUN_NAME = 'wishart-dictionary-100k'
PREPARED_OVERRIDE = None  # relative to DRIVE_ROOT or an absolute Drive path
DATASET_OVERRIDE = None   # relative to DRIVE_ROOT or an absolute Drive path

def drive_input_path(value):
    candidate = Path(value).expanduser()
    return candidate if candidate.is_absolute() else DRIVE_ROOT / candidate

prepared_path = None
dataset_path = None
if PREPARED_OVERRIDE is not None:
    prepared_path = drive_input_path(PREPARED_OVERRIDE)
elif DATASET_OVERRIDE is not None:
    dataset_path = drive_input_path(DATASET_OVERRIDE)
else:
    prepared_root = DRIVE_ROOT / 'prepared'
    completed = list(prepared_root.glob('*/COMPLETED')) if prepared_root.exists() else []
    if completed:
        completed.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        prepared_path = completed[0].parent
    else:
        candidate = DRIVE_ROOT / 'data' / 'conceptnet_en_100k.tsv'
        if candidate.is_file():
            dataset_path = candidate

if prepared_path is None and dataset_path is None:
    raise FileNotFoundError(
        f'No completed prepared graph or source TSV under {DRIVE_ROOT}. '
        'Set PREPARED_OVERRIDE/DATASET_OVERRIDE to an absolute existing Drive path '
        'or upload inputs under DRIVE_ROOT/prepared or DRIVE_ROOT/data.'
    )

DRIVE_RUN = DRIVE_ROOT / 'runs' / RUN_NAME
IS_COMPLETED = (DRIVE_RUN / 'COMPLETED').is_file()
RESUME = DRIVE_RUN.exists() and not IS_COMPLETED
if IS_COMPLETED:
    print('Run is already complete; inspection cells below can be used without rerunning.')
print({
    'git_revision': head,
    'config_drive': str(CONFIG_ON_DRIVE),
    'prepared': str(prepared_path) if prepared_path else None,
    'dataset': str(dataset_path) if dataset_path else None,
    'run_name': RUN_NAME,
    'resume': RESUME,
    'drive_output': str(DRIVE_RUN),
})


In [ ]:
# Main computation: Drive YAML + bounded CPU workers + CUDA when available.
# The CLI validates config/input/code identity before any resumed computation.
command = [
    'semmap-wishart-colab',
    '--config-drive', 'config/experiment.yaml',
    '--drive-root', str(DRIVE_ROOT),
    '--scratch-root', '/content/semmap-wishart',
    '--run-name', RUN_NAME,
    '--compare-mode', 'size_mtime',
    '--blas-threads', '1',
    '--device', 'auto',
]
if prepared_path is not None:
    command += ['--prepared-drive', str(prepared_path)]
else:
    command += ['--dataset-drive', str(dataset_path)]
if RESUME:
    command += ['--resume']

if IS_COMPLETED:
    print('Existing completed run:', DRIVE_RUN)
else:
    print('Run mode:', 'RESUME' if RESUME else 'NEW')
    subprocess.run(command, check=True)


In [ ]:
# Validate that this was a graph-dictionary run, not merely a legacy coarsening run.
required = [
    DRIVE_RUN / 'COMPLETED',
    DRIVE_RUN / 'hierarchy.json',
    DRIVE_RUN / 'dictionary' / 'graph_types.jsonl',
    DRIVE_RUN / 'dictionary' / 'grammar.jsonl',
    DRIVE_RUN / 'dictionary' / 'huffman.json',
    DRIVE_RUN / 'dictionary' / 'statistics.json',
    DRIVE_RUN / 'level_000' / 'symbolic_nodes.jsonl',
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise RuntimeError({'missing_dictionary_artifacts': missing})

hierarchy = json.loads((DRIVE_RUN / 'hierarchy.json').read_text(encoding='utf-8'))
stats = json.loads((DRIVE_RUN / 'dictionary' / 'statistics.json').read_text(encoding='utf-8'))
print('COMPLETED')
print(json.dumps({
    'initial_nodes': hierarchy['initial_nodes'],
    'final_nodes': hierarchy['final_nodes'],
    'levels': hierarchy['levels'],
    'stop_reason': hierarchy['stop_reason'],
    'dictionary_size': stats['dictionary_size'],
    'candidate_occurrences': stats['candidate_occurrences'],
    'accepted_occurrences': stats['accepted_occurrences'],
    'recursive_types': stats['recursive_types'],
}, indent=2))


In [ ]:
# Compact transition table for immediate experiment inspection.
import pandas as pd

rows = []
for tr in hierarchy.get('transitions', []):
    dm = tr.get('dictionary_metrics', {})
    rows.append({
        'level': tr.get('source_level'),
        'fine_nodes': tr.get('fine_nodes'),
        'coarse_nodes': tr.get('coarse_nodes'),
        'node_compression': tr.get('compression_ratio'),
        'dictionary_size': dm.get('dictionary_size_after_discovery'),
        'new_types': dm.get('new_types'),
        'reused_types': dm.get('reused_types'),
        'novelty': dm.get('dictionary_novelty'),
        'reuse_rate': dm.get('reuse_rate'),
        'recursive_types': dm.get('recursive_types_total'),
        'selected_occurrences': dm.get('selected_occurrences'),
        'mdl_gain_bits_proxy': dm.get('mdl_gain_bits_proxy'),
        'mdl_ratio_proxy': dm.get('mdl_ratio_proxy'),
        'entropy_bits': dm.get('type_entropy_bits'),
        'mean_huffman_bits': dm.get('mean_huffman_code_length'),
    })

table = pd.DataFrame(rows)
display(table)


In [ ]:
# Plot the four trajectories that matter most for the dictionary hypothesis.
import matplotlib.pyplot as plt

if not table.empty:
    for column in ['mdl_ratio_proxy', 'reuse_rate', 'novelty', 'recursive_types']:
        if column not in table or table[column].dropna().empty:
            continue
        plt.figure(figsize=(7, 4))
        plt.plot(table['level'], table[column], marker='o')
        plt.xlabel('compression level')
        plt.ylabel(column)
        plt.title(column)
        plt.grid(True, alpha=0.25)
        plt.show()


## Interpretation and persistence

The current MDL metric is a **structural coding proxy**, not the byte size of a completed lossless codec. Durable results, input manifests, and immutable per-level checkpoints are under `DRIVE_ROOT/runs/RUN_NAME`. Do not edit the YAML or switch Git revisions when resuming an interrupted run; start a new named run instead.
